# Teleportation
**Quantum Computing · Week 5 (Quantum protocols and algorithms) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w5d1.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Teleportation** moves an unknown qubit state from Alice to Bob with one shared Bell pair and two
classical bits. Build the circuit, run it, and check that Bob really ends up with Alice's state.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator

theta, phi = 1.1, 0.7                   # Alice's message: RY(theta), then a phase phi
q = QuantumRegister(3, "q")
m = ClassicalRegister(2, "m")           # Alice's two measurement results
qc = QuantumCircuit(q, m)
qc.ry(theta, 0)
qc.p(phi, 0)                            # q0 now holds the message |psi>
qc.barrier()
qc.h(1)
qc.cx(1, 2)                             # a Bell pair: q1 stays with Alice, q2 goes to Bob
qc.barrier()
qc.cx(0, 1)
qc.h(0)                                 # Alice's two gates...
qc.measure(0, m[0])
qc.measure(1, m[1])                     # ...and her two measurements
with qc.if_test((m[1], 1)):             # Bob's fixes, chosen by the two bits
    qc.x(2)
with qc.if_test((m[0], 1)):
    qc.z(2)
qc.draw("mpl", style="clifford")

## Did it work? The undo trick
Bob can't look at his qubit without disturbing it. But if teleportation worked, undoing Alice's
preparation on his qubit must give 0 every single time.

In [ ]:
check = qc.copy()
bob = ClassicalRegister(1, "bob")
check.add_register(bob)
check.p(-phi, 2)
check.ry(-theta, 2)                     # undo RY(theta) and P(phi), in reverse order
check.measure(2, bob[0])

sim = AerSimulator(seed_simulator=1)    # Aer can run measurements in the middle of a circuit
counts = sim.run(transpile(check, sim), shots=2000).result().get_counts()
alice, bob_counts = {}, {}
for key, n in counts.items():
    b, a = key.split()                  # registers are printed right to left: "bob m"
    alice[a] = alice.get(a, 0) + n
    bob_counts[b] = bob_counts.get(b, 0) + n
print("Alice's results (m1 m0):", dict(sorted(alice.items())))
print("Bob's undo test        :", bob_counts)

## Looking inside: the version without measurements
A controlled gate can stand in for "measure, then decide": a CNOT from q1 and a CZ from q0 make
Bob's fixes. This version runs on a statevector, so we can look at Bob's qubit directly with
`partial_trace`, which keeps one qubit and averages over the others.

In [ ]:
def message_state(theta, phi):
    msg = QuantumCircuit(1)
    msg.ry(theta, 0)
    msg.p(phi, 0)
    return Statevector(msg)

def teleport_unitary(theta, phi, fix=True):
    qc = QuantumCircuit(3)
    qc.ry(theta, 0)
    qc.p(phi, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    if fix:
        qc.cx(1, 2)                     # X on Bob's qubit when q1 is 1
        qc.cz(0, 2)                     # Z on Bob's qubit when q0 is 1
    return qc

bob_before = partial_trace(Statevector(teleport_unitary(theta, phi, fix=False)), [0, 1])
bob_after = partial_trace(Statevector(teleport_unitary(theta, phi)), [0, 1])
print("Bob's qubit before the fixes (a density matrix):")
print(np.round(bob_before.data.real, 3) + 0.0)        # + 0.0 turns -0.0 into 0.0
print("fidelity with Alice's message after the fixes:", round(state_fidelity(bob_after, message_state(theta, phi)), 6))

Before the fixes, Bob's qubit is the matrix $I/2$: a perfectly fair coin, whatever the message.
That's why teleportation can't send anything faster than light: until the two bits arrive, Bob has
nothing to read.

## Your turn: forget the Z fix
Bob only applies the X fix. Predict which messages still arrive correctly, then check the
fidelity for $|0\rangle$, $|1\rangle$, $|+\rangle$ and $|i\rangle$.

In [ ]:
def teleport_without_z(theta, phi):
    qc = QuantumCircuit(3)
    # TODO: the circuit of teleport_unitary, with only the CX fix (no CZ)
    return qc

messages = {"|0>": (0, 0), "|1>": (np.pi, 0), "|+>": (np.pi / 2, 0), "|i>": (np.pi / 2, np.pi / 2)}
for name, (t, p) in messages.items():
    bob_qubit = partial_trace(Statevector(teleport_without_z(t, p)), [0, 1])
    print(name, "fidelity:", round(state_fidelity(bob_qubit, message_state(t, p)), 3))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def teleport_without_z(theta, phi):
    qc = QuantumCircuit(3)
    qc.ry(theta, 0)
    qc.p(phi, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.cx(1, 2)                         # the X fix only
    return qc

for name, (t, p) in messages.items():
    bob_qubit = partial_trace(Statevector(teleport_without_z(t, p)), [0, 1])
    print(name, "fidelity:", round(state_fidelity(bob_qubit, message_state(t, p)), 3))
print("|0> and |1> survive: Z does nothing to them. |+> and |i> get a Z half of the time, so the fidelity drops to 0.5.")